# Session 7, Personas

> **A streaming service has 600 viewers and no idea who they are.** Find the segments, **name** them, and write **one brief** a marketing director could act on.

Claude writes the code. The names and the brief are yours, and they are what is marked.

## 0 · What you have

Six columns describe **what people do**. Two more describe **who they are**. That distinction matters more than it looks, and we come back to it in step 3.

In [ ]:
# --- Load the data -------------------------------------------------
DATA_URL = ""   # paste a hosted CSV link, or leave blank to upload netflix_viewers_full.csv
import pandas as pd
if DATA_URL:
    v = pd.read_csv(DATA_URL)
else:
    try:
        from google.colab import files
        print("Upload netflix_viewers_full.csv ...")
        v = pd.read_csv(next(iter(files.upload())))
    except Exception:
        v = pd.read_csv("data/netflix_viewers_full.csv")
v.head()

In [ ]:
BEHAVIOUR = ['hours_per_week', 'pct_series', 'pct_kids',
             'avg_session_mins', 'completion_rate', 'pct_new_releases']
ABOUT_THEM = ['age', 'n_profiles']
print(v[BEHAVIOUR].describe().round(2))

## 1 · Cluster

You know why the rescaling line is there. If you have forgotten, look at the ranges above and ask which column would win a distance calculation on its own.

**We cluster on behaviour only.** `age` and `n_profiles` are held back deliberately.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

Z = StandardScaler().fit_transform(v[BEHAVIOUR])
for k in range(3, 8):
    lab = KMeans(n_clusters=k, n_init=20, random_state=8).fit_predict(Z)
    print(f'k={k}  silhouette={silhouette_score(Z, lab):.3f}')

> **Your call.** Pick a k and be ready to say why. The silhouette is a hint, not an instruction, and Session 6 already taught you it does not get the last word.

In [ ]:
K = 5   # <- change this if you disagree
v['segment'] = KMeans(n_clusters=K, n_init=20, random_state=8).fit_predict(Z)
print(v['segment'].value_counts().sort_index())

## 2 · Profile

A segment mean on its own tells you nothing. **Compare it with everyone else.** A group that watches 14 hours is only interesting if the average is 4, or 40.

In [ ]:
overall = v[BEHAVIOUR].mean()
profile = v.groupby('segment')[BEHAVIOUR].mean()
print('Segment averages:'); print(profile.round(2))
print('\nDifference from the overall average:')
print((profile - overall).round(2))

> Ask Claude: *"For each segment, tell me the two features that differ most from the overall average, and phrase each one as a sentence about behaviour."* Then check it against the table yourself. It will occasionally describe a difference that is not there.

## 3 · Name

Now the part that is actually graded. Before you write a name down, run it through this test.

**Is this name about what they DO, or about who they ARE?**

Try it. Profile your segments on the two columns we held back:

In [ ]:
print(v.groupby('segment')[ABOUT_THEM].mean().round(1))
print('\nShare aged 55+ in each segment:')
print(v.groupby('segment').age.apply(lambda s: (s >= 55).mean()).round(2))

One of your segments is probably dominated by older viewers. **Age was never given to the algorithm.** It came out, it did not go in.

That is a fact about who happened to land in the group. It is not what the group *is*, and naming it that way changes what the business does to them, including the ones who do not fit.

> Write one name per segment. Behaviour, not identity.

In [ ]:
names = {
    0: '...',
    1: '...',
    2: '...',
    3: '...',
    4: '...',
}
v['persona'] = v['segment'].map(names)
v.groupby('persona')[BEHAVIOUR].mean().round(2)

## 4 · The brief

Pick **one** segment. Write half a page, four points:

1. **Who they are, in one line.** Behaviour, size, how they differ. No jargon, no mention of k.
2. **What they are worth.** How many people, and why they are worth spending money on.
3. **What to do differently.** One concrete action you would not do for the other segments.
4. **What would change your mind.** The number you would check in a month to find out you were wrong.

Point 4 is the one people skip and the one that separates a First. If you cannot say what would prove you wrong, you are not doing analytics.

> Ask Claude to critique your brief against these four points, then **disagree with it at least once in writing**. That exchange is worth keeping for your Claude log.

---
**Check before you finish:** would any two of your segments get the same action? If so, you have fewer segments than you think. Go back to step 1 and lower k.